# Dự đoán ngày mai tiệm bán bao nhiêu ly
Notebook đi kèm video 302 của kênh **Mark học AI** (series *Dự đoán bán hàng với AI*).
Dữ liệu: `so-ban-hang-mau.xlsx`, sổ bán hàng giả 3 tháng (7-9/2026) của tiệm trà sữa nhỏ, cùng file với video 301.

Cách dùng trên Google Colab: mở notebook, bấm biểu tượng thư mục bên trái và tải file `so-ban-hang-mau.xlsx` lên,
rồi chạy lần lượt từng ô (Shift + Enter). Đọc hiểu từng ô trước khi chạy.

## 1. Nạp file

In [ ]:
# Chưa tải file lên? Ô này tự lấy file mẫu từ markhocai.com (có rồi thì bỏ qua)
import os, urllib.request
if not os.path.exists("so-ban-hang-mau.xlsx"):
    urllib.request.urlretrieve("https://markhocai.com/assets/lam-theo/so-ban-hang-mau.xlsx", "so-ban-hang-mau.xlsx")

In [1]:
import pandas as pd

df = pd.read_excel("so-ban-hang-mau.xlsx")
df.head()

         Ngày    Giờ                 Món   SL Đơn giá  Thành tiền    SĐT khách     Ghi chú
0  01/07/2026   8:15   Trà sữa khoai môn  1.0   30000     30000.0          NaN  khách quen
1  01/07/2026   8:35          trà sữa TC  1.0   25000     25000.0          NaN         NaN
2  01/07/2026   9:49   Trà sữa khoai môn  1.0   30000     30000.0          NaN        grab
3  01/07/2026  10:36  Hồng trà macchiato  1.0   30000     30000.0  995308425.0     ít ngọt
4  01/07/2026  10:18   Trà sữa trân châu  1.0   25000     25000.0          NaN         NaN

In [2]:
print(len(df), "dòng")

7341 dòng


## 2. Dọn dữ liệu
Bỏ dòng trống và 3 dòng TỔNG THÁNG dì tự ghi (cái bẫy cộng trùng ở video 301), đưa cột Ngày về một định dạng.

In [3]:
df = df.dropna(subset=["Ngày"])
tong = df["Ngày"].astype(str).str.startswith("TỔNG")
df = df[~tong]
chu = df["Ngày"].astype(str).str.replace("-", "/")
chu = chu.str.replace(r"/(\d\d)$", r"/20\1", regex=True)
df["Ngày"] = pd.to_datetime(chu, format="%d/%m/%Y")
print(len(df), "dòng bán hàng")

7305 dòng bán hàng


## 3. Gom theo ngày

In [4]:
ngay = df.groupby("Ngày").agg(
    so_ly=("SL", "sum"),
    doanh_thu=("Thành tiền", "sum"),
).reset_index()
print(len(ngay), "ngày")
ngay.head()

92 ngày


        Ngày  so_ly  doanh_thu
0 2026-07-01   87.0  2530000.0
1 2026-07-02  103.0  3015000.0
2 2026-07-03   88.0  2545000.0
3 2026-07-04  147.0  4395000.0
4 2026-07-05  155.0  4630000.0

In [ ]:
# Vẽ số ly theo ngày (Colab có sẵn matplotlib)
ngay.plot(x="Ngày", y="so_ly", figsize=(12, 4), title="Số ly mỗi ngày")

## 4. Feature (đặc điểm) cho từng ngày
- `thu`: 0 = thứ Hai ... 6 = Chủ Nhật
- `cuoi_tuan`: 1 nếu thứ Bảy, Chủ Nhật
- `hom_qua`: số ly hôm qua
- `tuan_truoc`: số ly cùng thứ tuần trước (7 ngày trước)

In [6]:
ngay["thu"] = ngay["Ngày"].dt.dayofweek
ngay["cuoi_tuan"] = (ngay["thu"] >= 5).astype(int)
ngay["hom_qua"] = ngay["so_ly"].shift(1)
ngay["tuan_truoc"] = ngay["so_ly"].shift(7)
print(ngay.groupby("cuoi_tuan")["so_ly"].mean().round(1))

cuoi_tuan
0     96.5
1    137.3
Name: so_ly, dtype: float64


## 5. Chia dữ liệu: tháng 7-8 để học, tháng 9 để thi

In [7]:
du_lieu = ngay.dropna()
train = du_lieu[du_lieu["Ngày"] < "2026-09-01"]
test = du_lieu[du_lieu["Ngày"] >= "2026-09-01"]
print(len(train), "ngày để học,", len(test), "ngày để thi")

55 ngày để học, 30 ngày để thi


## 6. Baseline: đoán bằng cùng thứ tuần trước

In [8]:
from sklearn.metrics import mean_absolute_error

mae_baseline = mean_absolute_error(
    test["so_ly"], test["tuan_truoc"])
print("Baseline sai trung bình", round(mae_baseline, 1), "ly mỗi ngày")

Baseline sai trung bình 12.1 ly mỗi ngày


## 7. Model machine learning đầu tiên: linear regression

In [9]:
from sklearn.linear_model import LinearRegression

features = ["thu", "cuoi_tuan",
            "hom_qua", "tuan_truoc"]
model = LinearRegression()
model.fit(train[features], train["so_ly"])

du_doan = model.predict(test[features])
mae_model = mean_absolute_error(
    test["so_ly"], du_doan)
print("Model sai trung bình", round(mae_model, 1), "ly mỗi ngày")

Model sai trung bình 8.9 ly mỗi ngày


In [10]:
for ten, he_so in zip(features, model.coef_):
    print(ten, round(he_so, 2))

thu 0.42
cuoi_tuan 39.49
hom_qua 0.06
tuan_truoc -0.06


## 8. Bẫy 1: data leakage
Thử (SAI) thêm cột doanh thu của chính ngày cần đoán. Ngày mai chưa tới thì làm gì đã biết doanh thu ngày mai!

In [11]:
features_lo = features + ["doanh_thu"]
model_lo = LinearRegression()
model_lo.fit(train[features_lo], train["so_ly"])
mae_lo = mean_absolute_error(
    test["so_ly"], model_lo.predict(test[features_lo]))
print("Model có cột doanh thu sai trung bình", round(mae_lo, 1), "ly")

Model có cột doanh thu sai trung bình 1.4 ly


## 9. Bẫy 2: overfitting
Một model phức tạp hơn (cây quyết định không giới hạn độ sâu) học thuộc lòng dữ liệu học.

In [12]:
from sklearn.tree import DecisionTreeRegressor

cay = DecisionTreeRegressor(random_state=0)
cay.fit(train[features], train["so_ly"])
print("Cây: sai khi học", round(mean_absolute_error(train["so_ly"], cay.predict(train[features])), 1),
      "| sai khi thi", round(mean_absolute_error(test["so_ly"], cay.predict(test[features])), 1))
print("Linear regression: sai khi học", round(mean_absolute_error(train["so_ly"], model.predict(train[features])), 1),
      "| sai khi thi", round(mae_model, 1))

Cây: sai khi học 0.0 | sai khi thi 11.5
Linear regression: sai khi học 8.5 | sai khi thi 8.9


## 10. Dự đoán 7 ngày tới và đổi ra nguyên liệu
Học lại trên cả 3 tháng, đoán từng ngày; ngày sau dùng số vừa đoán làm *hôm qua*.

In [13]:
model.fit(du_lieu[features], du_lieu["so_ly"])

lich_su = list(ngay["so_ly"])
ket_qua = []
cac_ngay = pd.date_range("2026-10-01", periods=7)
for ngay_moi in cac_ngay:
    thu = ngay_moi.dayofweek
    x = pd.DataFrame([[thu, int(thu >= 5),
                       lich_su[-1], lich_su[-7]]],
                     columns=features)
    so_ly = model.predict(x)[0]
    lich_su.append(so_ly)
    ket_qua.append((ngay_moi.strftime("%d/%m"), round(so_ly)))
ket_qua

[('01/10', 97), ('02/10', 97), ('03/10', 136), ('04/10', 138), ('05/10', 98), ('06/10', 97), ('07/10', 95)]

In [14]:
# dự phòng mỗi ngày đúng bằng sai số trung bình
du_phong = round(mae_model)
tong_ly = sum(ly + du_phong for _, ly in ket_qua)
# định mức trung bình mỗi ly (sổ tay của dì)
TRAN_CHAU_G, SUA_ML = 40, 120
print("Chuẩn bị cho", tong_ly, "ly")
print("Trân châu:", round(tong_ly * TRAN_CHAU_G / 1000, 1), "kg")
print("Sữa:", round(tong_ly * SUA_ML / 1000, 1), "lít")

Chuẩn bị cho 821 ly
Trân châu: 32.8 kg
Sữa: 98.5 lít
